Task 1: File ingestion and regular expressions

In [351]:
import re
from pathlib import Path


def read_file_with_multiple_encodings(filename):
    path = Path(filename)

    for encoding in ['utf-8', 'latin-1']:
        try:
            text = path.read_text(encoding=encoding)
            return text, encoding
        except UnicodeDecodeError:
            continue

    raise ValueError("Could not decode file")


folder = Path("/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw")


log_pattern = r'\[(.*?)\] \[(.*?)\] item=(.*?) price=(\d+\.\d{2}) (EUR|NOK|USD) \((SUCCESS|REFUND|CANCELLED)\)'


def parse_log_files(file_content):
    sales = []
    for line in file_content.splitlines():
        match = re.match(log_pattern, line)

        if match:
            timestamp = match.group(1)
            register_id = match.group(2)
            item = match.group(3)
            price = float(match.group(4))
            currency = match.group(5)
            status = match.group(6)

            sale = {
            "timestamp": timestamp,
            "register_id": register_id,
            "item": item,
            "price": price,
            "currency": currency,
            "status": status
            }
            sales.append(sale)

    return sales


all_sales = []

for file_path in folder.rglob("*.log"):
    text, encoding = read_file_with_multiple_encodings(file_path)
    sales = parse_log_files(text)
    all_sales.extend(sales)

for i in range(5):
    print(all_sales[i])


{'timestamp': '2026-10-15 07:01:28', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.05, 'currency': 'USD', 'status': 'SUCCESS'}
{'timestamp': '2026-10-15 07:02:34', 'register_id': 'POS_01', 'item': 'Croissant', 'price': 3.47, 'currency': 'EUR', 'status': 'SUCCESS'}
{'timestamp': '2026-10-15 07:04:02', 'register_id': 'POS_01', 'item': 'Sandwich', 'price': 7.59, 'currency': 'NOK', 'status': 'SUCCESS'}
{'timestamp': '2026-10-15 07:04:33', 'register_id': 'POS_01', 'item': 'Muffin', 'price': 3.01, 'currency': 'USD', 'status': 'REFUND'}
{'timestamp': '2026-10-15 07:06:20', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.55, 'currency': 'EUR', 'status': 'SUCCESS'}


https://labex.io/tutorials/python-how-to-read-files-with-different-encodings-434794
https://www.geeksforgeeks.org/python/parse-and-clean-log-files-in-python/

Task 2: Recursive utility functions

In [156]:
import json
from pathlib import Path

path_to_dir = '/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw'


def find_log_files(input_path: str, path_list: list[Path] = []) -> list[pathlib.Path]:
    """Goes through a given directory and creates a list of all files with
    the .log suffix in that directory and its subdirectories"""
    dir_path = Path(input_path)
    for file in dir_path.iterdir():
        if file.is_dir():
            find_log_files(file, path_list)

        else:
            if file.suffix == '.log':
                path_list.append(file)

    return path_list


path_to_json = '/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/store_config.json'
with open(path_to_json) as json_file:
    json_data = json.load(json_file)


def traverse_json(data: dict | list, list_for_output: list[str] = []) -> set[str]:
    """Goes through a combination of nested dictionaries and lists and
    returns a set of register id strings where 'active': True"""
    if isinstance(data, dict):
        for key in data.keys():
            if key == 'active' and data[key] is True:
                list_for_output.append(data['id'])
            traverse_json(data[key])

    if isinstance(data, list):
        for index in range(len(data)):
            traverse_json(data[index])

    return set(list_for_output) 


for i in find_log_files(path_to_dir):
    print(i)

print(traverse_json(json_data))


/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/2026/science_building/POS_01/register_sales.log
/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/2026/science_building/POS_03/register_sales.log
/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/2026/science_building/POS_02/register_sales.log
/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/2026/student_union/POS_01/register_sales.log
/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/2026/student_union/POS_03/register_sales.log
/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/2026/student_union/POS_02/register_sales.log
/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/

Resource: 
https://docs.python.org/3/library/pathlib.html 
(For how to match the .log suffix, how .iterdir() works and how to check if path leads to a directory)

https://www.geeksforgeeks.org/python/read-json-file-using-python/
(For how to get the data in json file as list or dictionary)

https://mypy.readthedocs.io/en/stable/cheat_sheet_py3.html
(For help in making type hint for empty list)

Task 3: Functions for cleaning and analysing data using NumPy

In [ ]:
import numpy as np

def removing_price_outliers():
    price = [all_sales[i]["price"] for i in range(len(all_sales))]

    arr = np.array(price)
    mean = np.mean(arr)
    std = np.std(arr)

    detect_outliers = (abs(arr - mean) > 3*std)
    outliers = (arr[detect_outliers])
    median = np.median(arr[~detect_outliers])
    arr[detect_outliers] = median
    return arr

removing_price_outliers()


success= []
refund = []
all = []

def compute_refund_ratio():
    for i in all_sales:
        item = (i["item"])
        if i["status"] == "SUCCESS":
            success.append(item)
        if i["status"] == "REFUND":
            refund.append(item)
        all.append(len(success) + len(refund))

    print("Successfull sales:")
    array = np.array(success)
    sorts = np.sort(array)
    unique_words, counts = np.unique(sorts, return_counts=True)
    for item, count in zip(unique_words, counts):
        print(f"{item}: {count}" )

    print("-------------------------------------------")
    print("")
    print("Refunds:")

    array_refund = np.array(refund)
    sorts_refund = np.sort(array_refund)
    unique_words_refund, counts_refund = np.unique(sorts_refund, return_counts=True)
    for item, count in zip(unique_words_refund, counts_refund):
        print(f"{item}: {count}" )

    refund_ratio = counts_refund / (len(all) + 10**-6)*100
    print(refund_ratio)


compute_refund_ratio()


Successfull sales:
Cappuccino: 82
Croissant: 75
Espresso: 80
Latte: 84
Muffin: 72
Sandwich: 97
-------------------------------------------

Refunds:
Cappuccino: 17
Croissant: 19
Espresso: 25
Latte: 16
Muffin: 23
Sandwich: 17
[2.36111111 2.63888889 3.47222222 2.22222222 3.19444444 2.36111111]


https://www.geeksforgeeks.org/data-analysis/how-to-change-a-single-value-in-a-numpy-array/
https://numpy.org/devdocs/reference/generated/numpy.std.html

Task 4: Data analysis using Pandas

In [ ]:
import datetime as dt
import pandas as pd


def convert_to_dataframe(converting: list[dict]) -> pd.DataFrame:
    converting_df = pd.DataFrame(converting)
    converting_df['timestamp'] = pd.to_datetime(converting_df['timestamp'])
    converting_df.set_index('timestamp', inplace=True)
    sorted_df = converting_df.sort_index()
    return sorted_df


def hourly_revenue(input_df: pd.DataFrame) -> pd.Series:
    grouped = input_df.groupby('item')
    resampled = grouped['price'].resample('1h').sum()
    return resampled


def create_pivot_table(input_df: pd.DataFrame) -> pd.DataFrame:
    table = pd.pivot_table(input_df, values='price', index='item', columns='status', aggfunc='sum')
    return table


def refund_ratio_per_item(input_df: pd.DataFrame) -> None:
    input_array = input_df.to_numpy()
    success_list = []
    refund_list = []
    all_list = []

    for i in input_array:
        item = (i[1])
        if i[4] == "SUCCESS":
            success_list.append(item)
        if i[4] == "REFUND":
            refund_list.append(item)
        all_list.append(len(success_list) + len(refund_list))

    success_array = np.array(success_list)
    success_sorts = np.sort(success_array)
    success_unique_words, success_counts = np.unique(success_sorts, return_counts=True)

    refund_array = np.array(refund_list)
    refund_sorts = np.sort(refund_array)
    refund_unique_words, refund_counts = np.unique(refund_sorts, return_counts=True)

    return compute_refund_ratio(success_counts, refund_counts)


df = convert_to_dataframe(all_sales)

print(df.head())
print()
print(hourly_revenue(df).head())
print()
print(create_pivot_table(df))
print()
print(refund_ratio_per_item(df))

                    register_id      item  price currency   status
timestamp                                                         
2026-10-15 07:00:44      POS_03     Latte   4.34      NOK  SUCCESS
2026-10-15 07:00:59      POS_03  Espresso   2.98      EUR  SUCCESS
2026-10-15 07:01:08      POS_02  Espresso   3.16      NOK  SUCCESS
2026-10-15 07:01:23      POS_02  Sandwich   7.69      USD  SUCCESS
2026-10-15 07:01:28      POS_01     Latte   4.05      USD  SUCCESS

item        timestamp          
Cappuccino  2026-10-15 07:00:00    2209.21
            2026-10-15 08:00:00    1219.76
            2026-10-15 09:00:00    1027.80
Croissant   2026-10-15 07:00:00    1205.10
            2026-10-15 08:00:00    1175.61
Name: price, dtype: float64

status      CANCELLED   REFUND  SUCCESS
item                                   
Cappuccino      67.93    69.46  4319.38
Croissant     1058.10    72.54  2280.57
Espresso        78.68  1068.78  1239.18
Latte         1067.63    71.93  3359.66
Muffin        

Resource: https://www.geeksforgeeks.org/pandas/pandas-change-datatype/ (for how to convert dataframe values to datetime)

https://www.geeksforgeeks.org/python/how-to-change-index-values-in-pandas/ (For how to set as index in dataframe)

https://www.geeksforgeeks.org/pandas/how-to-sort-pandas-dataframe/ (For how to sort by index)

https://www.geeksforgeeks.org/pandas/python-pandas-dataframe-groupby/ (for how to group by item)

https://pandas.pydata.org/docs/dev/reference/api/pandas.DataFrame.resample.html (For how to resample)

Task 5: Create full workflow

In [10]:
# 1.
print('1. Recursively find all log files and extract active register IDs.')

dir_path = '/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw'
path_list = []

log_paths = find_log_files(dir_path, path_list)
for path in log_paths:
    print(path)

json_path = '/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/store_config.json'
with open(json_path) as json_file:
    data_json = json.load(json_file)

register_ids = traverse_json(data_json)
print(register_ids)
print()


# 2.
print('2. Ingest text logs and parse records into a list of dictionaries.')
all_sales2 = []

for path in log_paths:
    text2, encoding2 = read_file_with_multiple_encodings(path)
    sales2 = parse_log_files(text2)
    all_sales2.extend(sales2)

for i in range(6):
    print(all_sales2[i])
print()


# 3
print('3. Build the initial Pandas DataFrame')
initial_df = convert_to_dataframe(all_sales2)

print(initial_df.head())
print()


# 4
print('4. Apply NumPy outlier cleaning to remove 999.00 price glitches')
def removing_price_outliers(input_data: list) -> np.ndarray:
    price = [input_data["price"].iloc[i] for i in range(len(input_data))]
    arr = np.array(price)
    mean = np.mean(arr)
    std = np.std(arr)

    detect_outliers = (abs(arr - mean) > 3*std)
    outliers = (arr[detect_outliers])
    median = np.median(arr[~detect_outliers])
    arr[detect_outliers] = median
    return arr


cleaned_array = removing_price_outliers(initial_df)
cleaned_df = initial_df.copy()
cleaned_df["price"] = cleaned_array

print(cleaned_df.head())
print()


# 5
print('5. Apply currency correction')

new_currency = []
for i in range(len(cleaned_df)):
    price = cleaned_df['price'].iloc[i]
    currency = cleaned_df['currency'].iloc[i]
    if currency == "NOK":
        price = price * 10
    new_currency.append(price)

new_currency_df = cleaned_df.copy()
new_currency_df["price"] = new_currency

print(new_currency_df.head())
print()

# 6
print('6. Generate resampled reports, pivot tables and refund ratios.')
success = []
refund = []
all = []

for i in all_sales:
    item = (i["item"])
    if i["status"] == "SUCCESS":
        success.append(item)
    if i["status"] == "REFUND":
        refund.append(item)
    all.append(len(success) + len(refund))

array = np.array(success)
sorts = np.sort(array)
unique_words, counts = np.unique(sorts, return_counts=True)

array_refund = np.array(refund)
sorts_refund = np.sort(array_refund)
unique_words_refund, counts_refund = np.unique(sorts_refund, return_counts=True)


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    refund_ratio = refund_count / (sales_count + 10**-6)*100
    return refund_ratio


print(resampled_report(cleaned_df).head())
print()
print(create_pivot_table(cleaned_df))
print()
print(refund_ratio_per_item(cleaned_df))
print()


# 7. Save all output files
# Do it

1. Recursively find all log files and extract active register IDs.
/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/2026/science_building/POS_01/register_sales.log
/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/2026/science_building/POS_03/register_sales.log
/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/2026/science_building/POS_02/register_sales.log
/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/2026/student_union/POS_01/register_sales.log
/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/2026/student_union/POS_03/register_sales.log
/Users/norahattrem/Desktop/Programmering/Visual_Studio_Code/INF201/Deliverable2/Deliverable_2/Data/Raw/2026/student_union/POS_02/register_sales.log
/Users/norahattrem/Desktop/Programme

In [ ]:
# Task 3 for å fungere med task 4
import numpy as np


def removing_price_outliers(input_data: list) -> np.ndarray:
    price = [input_data[i]["price"] for i in range(len(input_data))]

    arr = np.array(price)
    mean = np.mean(arr)
    std = np.std(arr)

    detect_outliers = (abs(arr - mean) > 3*std)
    outliers = (arr[detect_outliers])
    median = np.median(arr[~detect_outliers])
    arr[detect_outliers] = median
    return arr


cleaned = removing_price_outliers(all_sales)


success = []
refund = []
all = []

for i in all_sales:
    item = (i["item"])
    if i["status"] == "SUCCESS":
        success.append(item)
    if i["status"] == "REFUND":
        refund.append(item)
    all.append(len(success) + len(refund))

array = np.array(success)
sorts = np.sort(array)
unique_words, counts = np.unique(sorts, return_counts=True)

array_refund = np.array(refund)
sorts_refund = np.sort(array_refund)
unique_words_refund, counts_refund = np.unique(sorts_refund, return_counts=True)


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    refund_ratio = refund_count / (sales_count + 10**-6)*100
    return refund_ratio


compute_refund_ratio(counts, counts_refund)